In [3]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

llm = ChatOpenAI(
    model="gpt-4.1-mini"
)

In [5]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    collection_name='my_documents',
    embedding_function=embeddings,
    persist_directory='./my_chroma_db'
)

In [6]:
mmr_retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 3, 'lambda_mult': 0.5}  # 'lambda_mult' : Relevance-Diversity Balance
)

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the provided context.

if the answer cannot be found in the context,
say "I don't know based on the provided document."

Context:
{context}

Question:
{question}

Answer:
""")

In [10]:
retrieve_chain = RunnableParallel({
    "context": mmr_retriever,
    "question": RunnablePassthrough()
})

In [12]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [13]:
rag_chain = retrieve_chain | prompt | llm | parser

In [18]:
question = """
Who was Daenerys Targaryen?
"""

result = rag_chain.invoke(question)

print(result)

Daenerys Targaryen, also called Daenerys Stormborn, the Unburnt, Mother of Dragons, Khaleesi of the Dothraki, and First of Her Name, was the sole surviving child of King Aerys II Targaryen by his sister/wife, Queen Rhaella. She was a widow at fourteen years and had newly hatched dragons named Drogon, Viserion, and Rhaegal.
